In [ ]:
# импортируем нужные библиотеки

# для работы с файлами формата .json
import json
# для упрощения отправки HTTP-запросов
import requests
# для генерации случайных чисел и строк (лучше, чем random)
import secrets
# Natural Language Toolkit (для получения слова)
import nltk
# для работы с временем
import time
# для работы с операционной системой
import os

In [ ]:
# эта функция реализует поиск определения слова в словаре Merriam-Webster

def get_definition(url):

    # вводим api-ключ
    api_key = 'e2f920d8-5454-43e7-befa-fe7fcaec4212'

    # получаем URL с помощью API
    try:
        url = url + "?key=" + api_key
        # отправляем запрос
        requested_page = requests.get(url, timeout=2)

        # проверяем статус страницы
        if requested_page.status_code != 200:
            return 'No Internet'

    except:
        return 'No Internet'

    # на случай, если слова нет в словаре
    try:
        page = requested_page.json()
    except:
        return None

    # если вернулся не список или список пуст, или первый элемент является строкой, пропускаем
    if not isinstance(page, list) or len(page) == 0 or isinstance(page[0], str):
        return None

    # безопасное извлечение определения (с)
    try:
        response = page[0]['shortdef'][0]
        return (response)

    except (KeyError, IndexError, TypeError, AttributeError) as e:
        return None

    return response

In [ ]:
# для работы с nltk следует предварительно установить  библиотеку в системе: pip install nltk

# скачиваем и импортируем wordnet
nltk.download('wordnet')
from nltk.corpus import wordnet as wn

# загружаем существительные
noun_synsets = list(wn.all_synsets('n'))
valid_nouns = []

def get_nouns():

    try:
        for syn in noun_synsets:
            for lemma in syn.lemmas():
                name = lemma.name()
                # применяем фильтр длины слова, наличия пробелов и дефисов
                if len(name) > 6 and '_' not in name and '-' not in name and name.islower():
                # добавляем в список
                    valid_nouns.append(name)
        return valid_nouns

    except:
        return 'No Internet'

In [ ]:
# создаём пару "слово-определение"
def get_random_pair():

    # пустой словарь для будущей пары
    random_pair = {}

    valid_nouns = get_nouns()

    # на случай отсутствия соединения
    if valid_nouns == "No Internet":
        return "No Internet"

    # получаем случайный элемент списка всех слов
    else:
        random_noun = secrets.choice(valid_nouns)

    # если получено пустое слово
    if not random_noun or random_noun.strip() == "":
        return None

    # создаём адрес запроса
    url  = "https://www.dictionaryapi.com/api/v3/references/collegiate/json/" + random_noun
    # получаем определение
    response = get_definition(url)

    # если нет связи
    if response == 'No Internet':
        return "No Internet"

    # получаем новые слова, пока не придёт непустой ответ без ненужных символов
    while response == 'ul' or response == None or response == '':

        random_noun = secrets.choice(valid_nouns)
        # формируем новый url
        url  = "https://www.dictionaryapi.com/api/v3/references/collegiate/json/" + random_noun
        # ждём, чтобы не перегружать api
        time.sleep(0.1)
        # получаем определение
        response = get_definition(url)

        # если нет связи
        if response == "No Internet":
            return "No Internet."

    # заполняем словарь
    random_pair[random_noun] = response.strip()

    return random_pair

In [ ]:
# проверяем наличие и наполнение файла со встроенной базой
try:

    if os.path.exists("base_installed_1.json") and os.path.getsize("base_installed_1.json") > 0:
        with open("base_installed_1.json", "r", encoding='utf-8') as infile:
            data = json.load(infile)

    else:
        data = {}
except:

    data = {}

count = 0

# замеряем время работы программы
start_general = time.time()


# генерируем нужное количество пар
while len(data) < 5000:

    random_pair = get_random_pair()

    # если пара пустая или вернулась строка, пропускаем
    if not random_pair or random_pair == 'No Internet' or isinstance(random_pair, str):
        continue

    # если определение пустое, пропускаем
    for key in random_pair:
        if not key or random_pair[key] == None or random_pair[key] == 'None':
            continue

    # если пара уже есть в словаре, пропускаем
    for key in random_pair:
        if key in data:
            continue

    for key, value in random_pair.items():
        length = len(data)
        
        #добавляем пару в словарь
        data[key] = value
        if len(data) == length + 1:
            count += 1

            if count == 1:
                print(f'{count} word collected...')

            else:
                print(f'{count} words collected...')

            if count % 100 == 0:
                end = time.time()
                print(f'Execution time: {round((end - start_general)) // 3600} h {round(((end - start_general) // 60) % 60)} min {round((end - start_general) % 60)} sec')


end_general = time.time()

print('Got', count, 'words!')

with open("base_installed_1.json", "w", encoding='utf-8') as outfile:
    json.dump(data, outfile, ensure_ascii=False)

In [ ]:
# проверяем наполняемость базы
print(len(data))

In [ ]:
# фиксируем время
end = time.time()

In [ ]:
# выводим время работы программы
print(f'Execution time: {round((end_general - start_general)) //3600} h {round((end_general - start_general)) // 60} min {round((end_general - start_general) % 60)} sec')

In [ ]:
# если нужно очистить базу

#with open('base_installed_1.json', 'w') as f:
#    json.dump({}, f)